# 02. Cible par échantillon et déséquilibre

In [ ]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # racine du projet
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from src import config, io, viz, models, features
from src import postprocess as pp, evaluation as ev
pd.set_option("display.width", 200)
print("Dossier données :", config.DATA_DIR)

Objectif : transformer les intervalles de crises en étiquettes échantillon/échantillon, puis par fenêtre.

> ℹ️ **Note d'architecture — Étiquetage & Fenêtrage :**
> La logique d'étiquetage par échantillon et par fenêtre est encapsulée dans src.labeling (sample_labels) et src.windowing (window_labels_from_samples). Une fenêtre de 2 s (50 % de recouvrement) est étiquetée comme « crise » si au moins 50 % de ses échantillons appartiennent à une crise annotée (config.LABEL_THRESHOLD = 0.5).

In [ ]:
seizures, _ = io.load_annotations()
mat = io.list_mat_files()[1]
rec = io.load_patient_signal(mat)
pid = io.patient_id_from_stem(mat.stem)
y = io.sample_labels(rec.sig.shape[0], rec.fs, seizures[seizures.Patient == pid])
from src.labeling import class_balance
print("Patient", pid, "déséquilibre :", class_balance(y))

Le signal est très déséquilibré (≈ 1-5 % de crises). C'est pourquoi on privilégie le **PR-AUC** et le **F1**, et non l'accuracy.

In [ ]:
fig = viz.plot_signal_overview(rec.t0, rec.names, rec.sig, rec.fs,
                               seizures[seizures.Patient == pid], title=f"{pid} — crises en rouge")
viz.save_fig(fig, "02_signal_overview"); plt.show()